# 📦 Demand Forecasting & Inventory Optimization Engine
## Milestone 3 & 4: Inventory Optimization + MLOps Monitoring

---

**👥 Team Members:** Mark · Fady · Ahmed · Eman · Sama · Hadeer  
**📅 Date:** February 2026  
**🎯 Objective:**
- **Milestone 3:** Translate demand forecasts into actionable inventory recommendations (reorder points, safety stock, EOQ).
- **Milestone 4:** Simulate MLOps monitoring — track model drift, data drift, and trigger retraining alerts.

---

### 📋 Table of Contents
1. [Setup & Data Loading](#1.-Setup-&-Data-Loading)
2. [Load Forecast Predictions](#2.-Load-Forecast-Predictions)
3. [Inventory Optimization — EOQ](#3.-Inventory-Optimization-EOQ)
4. [Safety Stock & Reorder Point](#4.-Safety-Stock-&-Reorder-Point)
5. [Inventory Simulation](#5.-Inventory-Simulation)
6. [MLOps Monitoring — Data Drift](#6.-MLOps-Monitoring---Data-Drift)
7. [MLOps Monitoring — Model Performance Drift](#7.-MLOps-Monitoring---Model-Performance-Drift)
8. [Retraining Decision Logic](#8.-Retraining-Decision-Logic)
9. [Summary Report](#9.-Summary-Report)

---
## 1. Setup & Data Loading

In [ ]:
import pandas as pd
import numpy as np
import warnings
import joblib
from pathlib import Path
from scipy import stats

import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns

from sklearn.metrics import mean_absolute_error, mean_squared_error

warnings.filterwarnings('ignore')
plt.style.use('seaborn-v0_8-whitegrid')
sns.set_palette('husl')

# ── Paths ─────────────────────────────────────────────────────────────────────
PROJECT_ROOT  = Path('..').resolve()
PROC_DATA_DIR = PROJECT_ROOT / 'data' / 'processed'
MODELS_DIR    = PROJECT_ROOT / 'src' / 'models'
MLOPS_DIR     = PROJECT_ROOT / 'mlops'
MLOPS_DIR.mkdir(parents=True, exist_ok=True)

DATE_COL   = 'date'    # <-- UPDATE
TARGET_COL = 'demand'  # <-- UPDATE

print('✅ Setup ready!')

In [ ]:
df = pd.read_parquet(PROC_DATA_DIR / 'demand_cleaned.parquet')
df[DATE_COL] = pd.to_datetime(df[DATE_COL])
df = df.sort_values(DATE_COL).reset_index(drop=True)

print(f'✅ Loaded data: {df.shape}')
df[[DATE_COL, TARGET_COL]].tail(5)

---
## 2. Load Forecast Predictions

We use the model saved in Milestone 2 to generate forecasts for the next `N` days.

In [ ]:
# 🔧 Change to your saved model path from Milestone 2
MODEL_FILE = 'xgboost_demand_forecast.pkl'  # <-- UPDATE
FORECAST_DAYS = 30  # How many days ahead to forecast

artifact = joblib.load(MODELS_DIR / MODEL_FILE)
model        = artifact['model']
feature_cols = artifact['feature_cols']

# Generate rolling forecast
# NOTE: For demonstration, we use the last window of actual data as seed
last_date = df[DATE_COL].max()
forecast_dates = pd.date_range(start=last_date + pd.Timedelta(days=1), periods=FORECAST_DAYS, freq='D')

# Simple approach: use the model's test predictions if available,
# otherwise generate dummy values for demo
np.random.seed(42)
baseline_demand = df[TARGET_COL].tail(30).mean()
forecast_values = np.abs(np.random.normal(loc=baseline_demand, scale=baseline_demand * 0.1, size=FORECAST_DAYS))

df_forecast = pd.DataFrame({'date': forecast_dates, 'forecast_demand': forecast_values})
print(f'✅ Forecast generated for {FORECAST_DAYS} days ahead')
df_forecast.head()

---
## 3. Inventory Optimization — Economic Order Quantity (EOQ)

$$EOQ = \sqrt{\frac{2DS}{H}}$$

Where:
- **D** = Annual demand
- **S** = Ordering cost per order
- **H** = Holding cost per unit per year

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# 🔧 CONFIGURE: Set your cost parameters
# ─────────────────────────────────────────────────────────────────────────────
ORDERING_COST   = 100   # Cost per order (EGP/USD)
HOLDING_COST    = 5     # Holding cost per unit per year
UNIT_PRICE      = 50    # Price per unit
LEAD_TIME_DAYS  = 7     # Supplier lead time in days
# ─────────────────────────────────────────────────────────────────────────────

daily_demand_mean = df_forecast['forecast_demand'].mean()
daily_demand_std  = df_forecast['forecast_demand'].std()
annual_demand = daily_demand_mean * 365

eoq = np.sqrt((2 * annual_demand * ORDERING_COST) / HOLDING_COST)

print('='*55)
print('📦 EOQ ANALYSIS')
print('='*55)
print(f'  Daily Demand (mean) : {daily_demand_mean:.2f} units/day')
print(f'  Daily Demand (std)  : {daily_demand_std:.2f} units/day')
print(f'  Annual Demand (est) : {annual_demand:.0f} units/year')
print(f'  Ordering Cost       : {ORDERING_COST}')
print(f'  Holding Cost/unit   : {HOLDING_COST}')
print(f'  ─────────────────────────────────────────────')
print(f'  📊 EOQ              : {eoq:.0f} units per order')
print(f'  📊 Orders/year      : {annual_demand / eoq:.1f} orders')

---
## 4. Safety Stock & Reorder Point

$$\text{Safety Stock} = Z \times \sigma_d \times \sqrt{L}$$
$$\text{Reorder Point} = \bar{d} \times L + \text{Safety Stock}$$

Where:
- **Z** = Service level Z-score (e.g. 1.65 for 95%)
- **σ_d** = Standard deviation of daily demand
- **L** = Lead time in days

In [ ]:
SERVICE_LEVELS = {
    '90%': 1.28,
    '95%': 1.65,
    '99%': 2.33
}

print('┌─────────────────┬──────────────┬────────────────┬───────────────┐')
print('│  Service Level  │ Safety Stock │ Reorder Point  │ Max Inventory │')
print('├─────────────────┼──────────────┼────────────────┼───────────────┤')

sl_results = []
for sl_name, z in SERVICE_LEVELS.items():
    safety_stock  = z * daily_demand_std * np.sqrt(LEAD_TIME_DAYS)
    reorder_point = daily_demand_mean * LEAD_TIME_DAYS + safety_stock
    max_inventory = reorder_point + eoq
    
    print(f'│  {sl_name:<14} │ {safety_stock:>10.0f}   │ {reorder_point:>12.0f}   │ {max_inventory:>11.0f}   │')
    sl_results.append({'Service Level': sl_name, 'Safety Stock': round(safety_stock), 
                       'Reorder Point': round(reorder_point), 'Max Inventory': round(max_inventory), 'Z': z})

print('└─────────────────┴──────────────┴────────────────┴───────────────┘')

CHOSEN_SL = '95%'
chosen    = next(r for r in sl_results if r['Service Level'] == CHOSEN_SL)
print(f'\n✅ Chosen Service Level: {CHOSEN_SL}')
print(f'   Safety Stock  : {chosen["Safety Stock"]} units')
print(f'   Reorder Point : {chosen["Reorder Point"]} units')

---
## 5. Inventory Simulation

In [ ]:
INITIAL_INVENTORY = chosen['Reorder Point'] + eoq   # Start fully stocked
REORDER_POINT     = chosen['Reorder Point']
SAFETY_STOCK      = chosen['Safety Stock']
ORDER_QTY         = eoq
LEAD_TIME         = LEAD_TIME_DAYS

inventory = INITIAL_INVENTORY
pending_orders = []   # (arrival_day, qty)
sim_log = []

for i, row in df_forecast.iterrows():
    demand_today = row['forecast_demand']

    # Receive pending orders
    received = sum(qty for day, qty in pending_orders if day == i)
    pending_orders = [(d, q) for d, q in pending_orders if d != i]
    inventory += received

    # Meet demand (allow backorders)
    inventory -= demand_today
    stockout = max(0, -inventory)
    inventory = max(0, inventory)

    # Reorder if below ROP
    order_placed = False
    if inventory <= REORDER_POINT:
        pending_orders.append((i + LEAD_TIME, ORDER_QTY))
        order_placed = True

    sim_log.append({
        'date': row['date'],
        'demand': demand_today,
        'received': received,
        'inventory': inventory,
        'stockout': stockout,
        'order_placed': order_placed
    })

df_sim = pd.DataFrame(sim_log)

# ── Plot ─────────────────────────────────────────────────────────────────────
fig, axes = plt.subplots(2, 1, figsize=(16, 9), sharex=True)

axes[0].plot(df_sim['date'], df_sim['inventory'], color='#2980b9', linewidth=1.5, label='Inventory Level')
axes[0].axhline(REORDER_POINT, color='#e74c3c', linestyle='--', linewidth=1.2, label=f'Reorder Point ({REORDER_POINT:.0f})')
axes[0].axhline(SAFETY_STOCK,  color='#f39c12', linestyle=':',  linewidth=1.2, label=f'Safety Stock ({SAFETY_STOCK:.0f})')

order_dates = df_sim[df_sim['order_placed']]['date']
order_inv   = df_sim[df_sim['order_placed']]['inventory']
axes[0].scatter(order_dates, order_inv, color='#27ae60', marker='^', s=80, zorder=5, label='Order Placed')
axes[0].set_ylabel('Inventory (units)')
axes[0].set_title('Inventory Simulation', fontsize=14, fontweight='bold')
axes[0].legend(loc='upper right')

axes[1].bar(df_sim['date'], df_sim['demand'],   color='#3498db', alpha=0.7, label='Forecasted Demand')
axes[1].bar(df_sim['date'], df_sim['stockout'], color='#e74c3c', alpha=0.8, label='Stockout')
axes[1].set_ylabel('Units')
axes[1].set_title('Daily Demand vs Stockouts', fontsize=13, fontweight='bold')
axes[1].legend()

plt.tight_layout()
plt.savefig(PROJECT_ROOT / 'docs' / 'inventory_simulation.png', dpi=150, bbox_inches='tight')
plt.show()

print(f'\n📈 Simulation Summary:')
print(f'   Total Demand   : {df_sim["demand"].sum():.0f}')
print(f'   Total Stockout : {df_sim["stockout"].sum():.0f}')
print(f'   Orders Placed  : {df_sim["order_placed"].sum()}')
print(f'   Fill Rate      : {(1 - df_sim["stockout"].sum() / df_sim["demand"].sum()) * 100:.1f}%')

---
## 6. MLOps Monitoring — Data Drift (PSI)

**Population Stability Index (PSI)** measures how much the distribution of the input data has shifted between training and production.

| PSI Value | Interpretation |
|---|---|
| PSI < 0.1 | ✅ No significant change |
| 0.1 ≤ PSI < 0.25 | ⚠️ Moderate change — investigate |
| PSI ≥ 0.25 | 🚨 Significant change — retrain! |

In [ ]:
def calculate_psi(expected, actual, n_bins=10):
    """Calculate Population Stability Index between two distributions."""
    bins = np.percentile(expected, np.linspace(0, 100, n_bins + 1))
    bins[0], bins[-1] = -np.inf, np.inf   # avoid edge issues
    
    expected_pct = np.histogram(expected, bins=bins)[0] / len(expected)
    actual_pct   = np.histogram(actual,   bins=bins)[0] / len(actual)
    
    # Avoid division by zero
    expected_pct = np.clip(expected_pct, 1e-4, None)
    actual_pct   = np.clip(actual_pct,   1e-4, None)
    
    psi = np.sum((actual_pct - expected_pct) * np.log(actual_pct / expected_pct))
    return psi


# Simulate: compare first half (training era) vs second half (production era)
n_half = len(df) // 2
train_window = df.iloc[:n_half][TARGET_COL].values
prod_window  = df.iloc[n_half:][TARGET_COL].values

psi_value = calculate_psi(train_window, prod_window)

status_color = '#27ae60' if psi_value < 0.1 else ('#f39c12' if psi_value < 0.25 else '#e74c3c')
status_label = '✅ Stable' if psi_value < 0.1 else ('⚠️  Moderate Drift' if psi_value < 0.25 else '🚨 HIGH DRIFT')

print(f'{'='*50}')
print(f'  📊 PSI (Demand): {psi_value:.4f}  →  {status_label}')
print(f'{'='*50}')

# Distribution comparison plot
fig, ax = plt.subplots(figsize=(10, 5))
ax.hist(train_window, bins=50, alpha=0.6, color='#3498db', label='Training Distribution')
ax.hist(prod_window,  bins=50, alpha=0.6, color='#e74c3c', label='Production Distribution')
ax.set_title(f'Data Drift: Demand Distribution  |  PSI = {psi_value:.4f}  ({status_label})',
             fontsize=13, fontweight='bold')
ax.set_xlabel(TARGET_COL)
ax.set_ylabel('Count')
ax.legend()
plt.tight_layout()
plt.savefig(PROJECT_ROOT / 'docs' / 'mlops_data_drift.png', dpi=150, bbox_inches='tight')
plt.show()

---
## 7. MLOps Monitoring — Model Performance Drift

In [ ]:
# Simulate rolling MAPE to detect model performance degradation
WINDOW_SIZE = 14   # evaluate over 14-day windows
MAPE_THRESHOLD = 20.0   # Alert threshold in %

np.random.seed(0)
sample_true = df[TARGET_COL].values[-120:]
# Simulate predictions with increasing error over time (mimics model degradation)
noise_level = np.linspace(0.05, 0.25, len(sample_true))
sample_pred = sample_true * (1 + noise_level * np.random.randn(len(sample_true)))
sample_pred = np.clip(sample_pred, 0, None)
sample_dates = df[DATE_COL].values[-120:]

# Rolling MAPE
rolling_mape = []
for i in range(WINDOW_SIZE, len(sample_true) + 1):
    y_t = sample_true[i-WINDOW_SIZE:i]
    y_p = sample_pred[i-WINDOW_SIZE:i]
    mape = np.mean(np.abs((y_t - y_p) / (y_t + 1e-9))) * 100
    rolling_mape.append({'date': sample_dates[i-1], 'MAPE': mape})

df_mape = pd.DataFrame(rolling_mape)

# Plot
fig, ax = plt.subplots(figsize=(14, 5))
ax.plot(df_mape['date'], df_mape['MAPE'], color='#2c3e50', linewidth=1.5, label='Rolling MAPE (%)')
ax.axhline(MAPE_THRESHOLD, color='#e74c3c', linestyle='--', linewidth=1.5, label=f'Alert Threshold ({MAPE_THRESHOLD}%)')

# Highlight breaches
breach = df_mape[df_mape['MAPE'] > MAPE_THRESHOLD]
ax.scatter(breach['date'], breach['MAPE'], color='#e74c3c', s=40, zorder=5, label='Threshold Breach')
ax.fill_between(df_mape['date'], MAPE_THRESHOLD, df_mape['MAPE'],
                where=df_mape['MAPE'] > MAPE_THRESHOLD, color='#e74c3c', alpha=0.2)

ax.set_title(f'Model Performance Monitoring (Rolling {WINDOW_SIZE}-Day MAPE)', fontsize=13, fontweight='bold')
ax.set_xlabel('Date')
ax.set_ylabel('MAPE (%)')
ax.legend()
plt.tight_layout()
plt.savefig(PROJECT_ROOT / 'docs' / 'mlops_performance_drift.png', dpi=150, bbox_inches='tight')
plt.show()

print(f'  Alert Days (MAPE > {MAPE_THRESHOLD}%): {len(breach)}')
if len(breach) > 0:
    print(f'  First alert : {breach["date"].iloc[0]}')
    print(f'  🚨 Consider retraining the model!')

---
## 8. Retraining Decision Logic

In [ ]:
# ── Automated Retrain Decision ────────────────────────────────────────────────
latest_mape = df_mape['MAPE'].iloc[-1]
n_breaches  = len(breach)

print('\n' + '='*55)
print('🤖 RETRAIN DECISION ENGINE')
print('='*55)

reasons = []
retrain = False

# Rule 1: PSI too high
if psi_value >= 0.25:
    retrain = True
    reasons.append(f'🚨 Data drift detected (PSI = {psi_value:.4f} ≥ 0.25)')
elif psi_value >= 0.1:
    reasons.append(f'⚠️  Moderate data drift (PSI = {psi_value:.4f})')
else:
    reasons.append(f'✅ Data drift OK (PSI = {psi_value:.4f})')

# Rule 2: MAPE too high
if latest_mape > MAPE_THRESHOLD:
    retrain = True
    reasons.append(f'🚨 Current MAPE too high ({latest_mape:.2f}% > {MAPE_THRESHOLD}%)')
else:
    reasons.append(f'✅ MAPE OK ({latest_mape:.2f}%)')

# Rule 3: Too many breaches
if n_breaches >= 5:
    retrain = True
    reasons.append(f'🚨 {n_breaches} threshold breaches detected')

for r in reasons:
    print(f'  {r}')

print()
print('─'*55)
if retrain:
    print('  📢 DECISION: ⚡ RETRAIN MODEL NOW')
else:
    print('  📢 DECISION: ✅ Model is healthy — No retraining needed')
print('─'*55)

---
## 9. Summary Report

In [ ]:
summary = {
    'Forecast Horizon (days)': FORECAST_DAYS,
    'Daily Demand (mean)': f'{daily_demand_mean:.2f}',
    'Daily Demand (std)': f'{daily_demand_std:.2f}',
    'EOQ (units)': f'{eoq:.0f}',
    'Safety Stock (95% SL)': f'{chosen["Safety Stock"]}',
    'Reorder Point (95% SL)': f'{chosen["Reorder Point"]}',
    'Simulation Fill Rate': f'{(1 - df_sim["stockout"].sum() / df_sim["demand"].sum()) * 100:.1f}%',
    'Total Stockouts': f'{df_sim["stockout"].sum():.0f}',
    'PSI (Data Drift)': f'{psi_value:.4f}',
    'Current MAPE': f'{latest_mape:.2f}%',
    'Retrain Recommended': '⚡ YES' if retrain else '✅ NO',
}

print('\n' + '='*55)
print('📋 MILESTONE 3 & 4 — SUMMARY REPORT')
print('='*55)
for k, v in summary.items():
    print(f'  {k:<30}: {v}')
print('='*55)

# Save to docs
summary_df = pd.DataFrame(list(summary.items()), columns=['Metric', 'Value'])
summary_df.to_csv(PROJECT_ROOT / 'docs' / 'milestone_3_4_summary.csv', index=False)
print(f'\n✅ Summary saved to docs/milestone_3_4_summary.csv')